# Item co-visitation matrix

Issue: #7

## Question

Which items are viewed, carted or ordered together in the same session? This matrix is the strong non-learned baseline (#18) and the base for the synthetic categories (#8).

## Data used

Input: `data/sample/events.parquet`, **training window w0 only** (weeks 1-2, days 0-13, from `split.py` in #6).

Output: `data/covis/covis.npz` (sparse matrix), `data/covis/item_ids.npy` (row/column i is item `item_ids[i]`), `data/covis/meta.json`.

Code: `notebooks/covis.py` (`from covis import build_covisitation`), which stays out of `src/ranking/` until #52.

## Definition (choices written down)

- **Pair:** two events in one session, the second at most **10 events** after the first, at most **24 hours** later, and different items.
- **Weight:** `(w[type_a] + w[type_b]) / 2 * 0.5 ** (gap / 6 hours)`, with click 1, cart 3, order 6. Stronger actions and closer events count more.
- **Symmetric:** pair (a, b) counts for a to b and b to a. We only ask which items go together, not which came first.
- **Minimum frequency:** items with fewer than **5 events** in the training window are dropped before pairing (no row or column).
- **Neighbour cap:** each item keeps its **100 strongest** neighbours. This makes the matrix small enough for 8GB but no longer exactly symmetric.

In [ ]:
import json
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from covis import DAY_MS, HOUR_MS, build_covisitation
from split import WINDOWS, day_to_ms

DATA = Path("data/sample") if Path("data/sample").exists() else Path("../data/sample")
OUT = DATA.parent / "covis"
OUT.mkdir(parents=True, exist_ok=True)


def peak_gb():
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9

## A toy example with hand-calculated weights

Times in hours. Types: 0 click, 1 cart, 2 order. Use `window=3`, `max_gap = 24 h`, half-life 6 h, no minimum frequency and no cap.

| Session | Events (item, type, hour) |
|---|---|
| S1 | A click 0, B cart 6, C order 12 |
| S2 | A click 0, B click 0 |
| S3 | A click 0, A click 1, D click 30 |

Hand calculation:
- S1 A-B: gap 6 h, (1+3)/2 = 2, times 0.5^(6/6) = 0.5, so **1.0**
- S1 A-C: gap 12 h, (1+6)/2 = 3.5, times 0.5^2 = 0.25, so **0.875**
- S1 B-C: gap 6 h, (3+6)/2 = 4.5, times 0.5, so **2.25**
- S2 A-B: gap 0, (1+1)/2 = 1, times 1, so **1.0**. A-B total is 1.0 + 1.0 = **2.0**
- S3: A-A is the same item and is skipped. A-D has a 29 h gap (more than 24 h) and is skipped.

In [ ]:
H = HOUR_MS
A, B, Cc, D = 10, 20, 30, 40  # item ids
session = np.array([1, 1, 1, 2, 2, 3, 3, 3])
aid = np.array([A, B, Cc, A, B, A, A, D])
etype = np.array([0, 1, 2, 0, 0, 0, 0, 0])
ts = np.array([0, 6, 12, 0, 0, 0, 1, 30]) * H

matrix, item_ids = build_covisitation(
    session, aid, ts, etype, min_freq=1, window=3, max_gap_ms=DAY_MS, halflife_ms=6 * H, top_k=None
)
assert list(item_ids) == [A, B, Cc, D]
dense = matrix.toarray()
expected = np.zeros((4, 4))
expected[0, 1] = expected[1, 0] = 2.0      # A-B
expected[0, 2] = expected[2, 0] = 0.875    # A-C
expected[1, 2] = expected[2, 1] = 2.25     # B-C
assert np.allclose(dense, expected, atol=1e-6), dense
assert dense[3].sum() == 0                  # D has no neighbours
assert np.allclose(np.diag(dense), 0)       # no self pairs
print(dense)

# neighbour cap: with top_k=1 item A keeps only its strongest neighbour, B (2.0 > 0.875)
capped, _ = build_covisitation(
    session, aid, ts, etype, min_freq=1, window=3, max_gap_ms=DAY_MS, halflife_ms=6 * H, top_k=1
)
assert capped.toarray()[0].tolist() == [0.0, 2.0, 0.0, 0.0]
# minimum frequency: item D (1 event) and C (1 event) are dropped when min_freq=2
small, small_ids = build_covisitation(
    session, aid, ts, etype, min_freq=2, window=3, max_gap_ms=DAY_MS, halflife_ms=6 * H, top_k=None
)
assert list(small_ids) == [A, B] and small.shape == (2, 2)
print("toy example passes")

## Build it on the training window

In [ ]:
w = WINDOWS["w0 initial (weeks 1-2)"]
train_end = day_to_ms(w["train_end_day"])
cols = (
    pl.scan_parquet(DATA / "events.parquet")
    .filter(pl.col("ts") < train_end)
    .select("session", "aid", "ts", "type")
    .collect()
)
session, aid, ts, etype = (cols[c].to_numpy() for c in ("session", "aid", "ts", "type"))
assert ts.max() < train_end, "the matrix must be built from the training window only"
print(f"training window: {len(ts):,} events, {len(np.unique(session)):,} sessions, latest ts {ts.max()} < {train_end}")

start = time.time()
matrix, item_ids = build_covisitation(session, aid, ts, etype, min_freq=5, window=10, top_k=100)
print(f"built in {time.time() - start:.0f} s, peak memory so far {peak_gb():.2f} GB")

In [ ]:
n = matrix.shape[0]
print("shape:", matrix.shape)
print("non-zero entries:", f"{matrix.nnz:,}")
print("density:", f"{matrix.nnz / n / n:.6f}")
print("items dropped by min frequency:", f"{len(np.unique(aid)) - n:,} of {len(np.unique(aid)):,}")
per_row = np.diff(matrix.indptr)
print("neighbours per item: median", int(np.median(per_row)), " max", int(per_row.max()), " items with none", int((per_row == 0).sum()))
assert per_row.max() <= 100
assert matrix.shape[0] == matrix.shape[1] == len(item_ids)
assert (matrix.diagonal() == 0).all()
assert (matrix.data > 0).all()

In [ ]:
sparse.save_npz(OUT / "covis.npz", matrix)
np.save(OUT / "item_ids.npy", item_ids)
meta = {
    "window": "w0 initial (weeks 1-2)",
    "train_end_ms": int(train_end),
    "min_freq": 5, "window_events": 10, "max_gap_hours": 24, "halflife_hours": 6, "top_k": 100,
    "type_weights": {"click": 1, "cart": 3, "order": 6},
    "items": int(n), "nnz": int(matrix.nnz),
}
(OUT / "meta.json").write_text(json.dumps(meta, indent=2))

# reload and check
again = sparse.load_npz(OUT / "covis.npz")
assert (again != matrix).nnz == 0
assert (np.load(OUT / "item_ids.npy") == item_ids).all()
print("saved and reloaded:", sorted(p.name for p in OUT.iterdir()))

In [ ]:
print(f"peak memory of this notebook: {peak_gb():.2f} GB")
assert peak_gb() < 6

## Result

The toy example matches the hand calculation. The real matrix is built from the training window only (checked: latest timestamp is before the end of day 13) and saved with its item ID mapping.

## What I learned

To be written by the owner of the project.